In [ ]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

db_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(db_url)

job_postings_fact = pd.read_sql_query(
    sql=text("SELECT * FROM data_jobs.job_postings_fact;"),
    con=engine,
)

company_dim = pd.read_sql_query(
    sql=text("SELECT * FROM data_jobs.company_dim;"),
    con=engine,
)

skills_dim = pd.read_sql_query(
    sql=text("SELECT * FROM data_jobs.skills_dim;"),
    con=engine,
)

skills_job_dim = pd.read_sql_query(
    sql=text("SELECT * FROM data_jobs.skills_job_dim;"),
    con=engine,
)

dataframes = {
    "company_dim": company_dim,
    "job_postings_fact": job_postings_fact,
    "skills_dim": skills_dim,
    "skills_job_dim": skills_job_dim,
}

print("Loaded DataFrames:")
for table_name, dataframe in dataframes.items():
    print(f"- {table_name}: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns")

Loaded DataFrames:
- company_dim: 98,372 rows × 5 columns
- job_postings_fact: 478,895 rows × 16 columns
- skills_dim: 254 rows × 3 columns
- skills_job_dim: 2,274,756 rows × 2 columns


In [7]:
ANALYSIS_START = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
ANALYSIS_END = pd.Timestamp("2025-01-01 00:00:00", tz="UTC")

date_scope_summary = pd.DataFrame(
    {
        "metric": [
            "source_postings_total",
            "minimum_posted_datetime_utc",
            "maximum_posted_datetime_utc",
            "postings_before_2024_utc",
            "postings_in_2024_utc",
            "postings_from_2025_utc",
            "share_in_2024_utc_pct",
            "analysis_postings_total",
        ],
        "value": [
            len(job_postings_fact),
            job_postings_fact["job_posted_date"].min(),
            job_postings_fact["job_posted_date"].max(),
            (job_postings_fact["job_posted_date"] < ANALYSIS_START).sum(),
            (
                (job_postings_fact["job_posted_date"] >= ANALYSIS_START)
                & (job_postings_fact["job_posted_date"] < ANALYSIS_END)
            ).sum(),
            (job_postings_fact["job_posted_date"] >= ANALYSIS_END).sum(),
            round(
                (
                    (
                        (job_postings_fact["job_posted_date"] >= ANALYSIS_START)
                        & (job_postings_fact["job_posted_date"] < ANALYSIS_END)
                    ).mean()
                    * 100
                ),
                2,
            ),
            len(job_postings_fact),
        ],
    }
)

display(date_scope_summary)

job_postings_analysis = job_postings_fact.copy()

print(f"\nRows retained for analysis: {len(job_postings_analysis):,}")
print(
    "Decision: all source postings are retained; "
    "90 boundary records before 2024 UTC are documented rather than excluded."
)

,metric,value
0,source_postings_total,478895
1,minimum_posted_datetime_utc,2023-12-31 23:00:01+00:00
2,maximum_posted_datetime_utc,2024-12-31 22:37:38+00:00
3,postings_before_2024_utc,90
4,postings_in_2024_utc,478805
5,postings_from_2025_utc,0
6,share_in_2024_utc_pct,99.98
7,analysis_postings_total,478895



Rows retained for analysis: 478,895
Decision: all source postings are retained; 90 boundary records before 2024 UTC are documented rather than excluded.


In [8]:
missingness_profile = (
    job_postings_analysis.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missingness_profile["total_rows"] = len(job_postings_analysis)

missingness_profile["missing_pct"] = (
    missingness_profile["missing_count"]
    / missingness_profile["total_rows"]
    * 100
).round(2)

missingness_profile["non_missing_count"] = (
    missingness_profile["total_rows"]
    - missingness_profile["missing_count"]
)

missingness_profile = (
    missingness_profile
    .sort_values(
        by=["missing_pct", "missing_count"],
        ascending=False,
    )
    .reset_index(names="column_name")
)

display(missingness_profile)

,column_name,missing_count,total_rows,missing_pct,non_missing_count
0,salary_hour_avg,468819,478895,97.90,10076
1,salary_year_avg,458560,478895,95.75,20335
2,salary_rate,448484,478895,93.65,30411
3,job_schedule_type,7912,478895,1.65,470983
4,job_location,1373,478895,0.29,477522
5,job_country,321,478895,0.07,478574
6,job_via,6,478895,0.00,478889
7,job_title,1,478895,0.00,478894
8,job_id,0,478895,0.00,478895
9,company_id,0,478895,0.00,478895


In [5]:
duplicate_records_summary = []

for table_name, dataframe in dataframes.items():
    total_rows = len(dataframe)
    duplicate_rows = dataframe.duplicated().sum()
    
    duplicate_records_summary.append(
        {
            "table": table_name,
            "total_rows": total_rows,
            "fully_duplicated_rows": duplicate_rows,
            "fully_duplicated_rows_pct": round(
                duplicate_rows / total_rows * 100,
                4,
            ),
        }
    )

duplicate_records_summary = pd.DataFrame(duplicate_records_summary)

display(duplicate_records_summary)

,table,total_rows,fully_duplicated_rows,fully_duplicated_rows_pct
0,company_dim,98372,0,0.00
1,job_postings_fact,478895,0,0.00
2,skills_dim,254,0,0.00
3,skills_job_dim,2274756,0,0.00


In [9]:
categorical_columns = [
    "job_title_short",
    "job_country",
    "job_schedule_type",
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance",
    "salary_rate",
]

categorical_profiles = {}

for column_name in categorical_columns:
    profile = (
        job_postings_analysis[column_name]
        .fillna("Missing")
        .value_counts(dropna=False)
        .rename_axis(column_name)
        .reset_index(name="posting_count")
    )

    profile["posting_share_pct"] = (
        profile["posting_count"]
        / len(job_postings_analysis)
        * 100
    ).round(2)

    categorical_profiles[column_name] = profile

print("Job role distribution:")
display(categorical_profiles["job_title_short"])

print("\nTop 20 countries:")
display(categorical_profiles["job_country"].head(20))

print("\nSchedule type distribution:")
display(categorical_profiles["job_schedule_type"].head(20))

print("\nWork-from-home flag distribution:")
display(categorical_profiles["job_work_from_home"])

print("\nNo-degree-mention flag distribution:")
display(categorical_profiles["job_no_degree_mention"])

print("\nHealth-insurance flag distribution:")
display(categorical_profiles["job_health_insurance"])

print("\nSalary-rate availability and type:")
display(categorical_profiles["salary_rate"])

Job role distribution:


,job_title_short,posting_count,posting_share_pct
0,Data Engineer,128994,26.94
1,Data Analyst,112866,23.57
2,Data Scientist,97664,20.39
3,Senior Data Engineer,30608,6.39
4,Business Analyst,28584,5.97
5,Software Engineer,23402,4.89
6,Senior Data Scientist,21731,4.54
7,Senior Data Analyst,15347,3.20
8,Machine Learning Engineer,12860,2.69
9,Cloud Engineer,6839,1.43



Top 20 countries:


,job_country,posting_count,posting_share_pct
0,United States,140365,29.31
1,India,39263,8.20
2,United Kingdom,35684,7.45
3,France,22562,4.71
4,Germany,18869,3.94
5,Spain,13033,2.72
6,Canada,11754,2.45
7,Netherlands,10952,2.29
8,Sudan,10688,2.23
9,Singapore,10051,2.10



Schedule type distribution:


...

,job_no_degree_mention,posting_count,posting_share_pct
0,False,323175,67.48
1,True,155720,32.52



Health-insurance flag distribution:


,job_health_insurance,posting_count,posting_share_pct
0,False,417452,87.17
1,True,61443,12.83



Salary-rate availability and type:


,salary_rate,posting_count,posting_share_pct
0,Missing,448484,93.65
1,year,20335,4.25
2,hour,10076,2.10


In [10]:
location_audit = pd.DataFrame(
    {
        "metric": [
            "total_postings",
            "missing_job_location",
            "missing_job_location_pct",
            "non_missing_job_location",
            "unique_non_missing_job_locations",
            "anywhere_exact_match",
            "anywhere_exact_match_pct",
            "locations_containing_remote_case_insensitive",
            "locations_containing_remote_pct",
        ],
        "value": [
            len(job_postings_analysis),
            job_postings_analysis["job_location"].isna().sum(),
            round(
                job_postings_analysis["job_location"].isna().mean() * 100,
                2,
            ),
            job_postings_analysis["job_location"].notna().sum(),
            job_postings_analysis["job_location"].nunique(dropna=True),
            job_postings_analysis["job_location"].eq("Anywhere").sum(),
            round(
                job_postings_analysis["job_location"].eq("Anywhere").mean() * 100,
                2,
            ),
            job_postings_analysis["job_location"].fillna("").str.contains(
                "remote",
                case=False,
                regex=False,
            ).sum(),
            round(
                job_postings_analysis["job_location"].fillna("").str.contains(
                    "remote",
                    case=False,
                    regex=False,
                ).mean() * 100,
                2,
            ),
        ],
    }
)

display(location_audit)

top_job_locations = (
    job_postings_analysis["job_location"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("job_location")
    .reset_index(name="posting_count")
)

top_job_locations["posting_share_pct"] = (
    top_job_locations["posting_count"]
    / len(job_postings_analysis)
    * 100
).round(2)

display(top_job_locations.head(50))

,metric,value
0,total_postings,"478,895.00"
1,missing_job_location,"1,373.00"
2,missing_job_location_pct,0.29
3,non_missing_job_location,"477,522.00"
4,unique_non_missing_job_locations,"16,360.00"
5,anywhere_exact_match,"63,090.00"
6,anywhere_exact_match_pct,13.17
7,locations_containing_remote_case_insensitive,621.00
8,locations_containing_remote_pct,0.13


,job_location,posting_count,posting_share_pct
0,Anywhere,63090,13.17
1,Singapore,9761,2.04
2,"New York, NY",8581,1.79
3,United Kingdom,7329,1.53
4,"Bengaluru, Karnataka, India",5840,1.22
5,"Paris, France",5758,1.20
6,India,5123,1.07
7,"Hyderabad, Telangana, India",4875,1.02
8,"Madrid, Spain",4847,1.01
9,"London, UK",4812,1.00


In [11]:
salary_columns = [
    "salary_year_avg",
    "salary_hour_avg",
]

salary_quality_profile = []

for column_name in salary_columns:
    available_salary = job_postings_analysis[column_name].dropna()

    salary_quality_profile.append(
        {
            "salary_column": column_name,
            "non_missing_count": len(available_salary),
            "coverage_pct": round(
                len(available_salary)
                / len(job_postings_analysis)
                * 100,
                2,
            ),
            "minimum": available_salary.min(),
            "p01": available_salary.quantile(0.01),
            "p05": available_salary.quantile(0.05),
            "median": available_salary.median(),
            "mean": available_salary.mean(),
            "p95": available_salary.quantile(0.95),
            "p99": available_salary.quantile(0.99),
            "maximum": available_salary.max(),
            "zero_or_negative_count": (available_salary <= 0).sum(),
        }
    )

salary_quality_profile = pd.DataFrame(salary_quality_profile)

display(salary_quality_profile)

,salary_column,non_missing_count,coverage_pct,minimum,p01,p05,median,mean,p95,p99,maximum,zero_or_negative_count
0,salary_year_avg,20335,4.25,"15,000.00","37,500.00","56,500.00","113,250.00","120,587.93","209,000.00","255,000.00","920,000.00",0
1,salary_hour_avg,10076,2.10,8.00,15.00,20.69,47.62,49.16,82.50,200.00,250.00,0


In [12]:
data_limitations = pd.DataFrame(
    {
        "area": [
            "Temporal scope",
            "Salary coverage",
            "Hourly salary coverage",
            "Location completeness",
            "Location standardization",
            "Company mapping",
            "Remote-work interpretation",
            "Dataset representation",
        ],
        "observation": [
            (
                f"{(job_postings_analysis['job_posted_date'] < ANALYSIS_START).sum():,} postings "
                "are dated before 2024-01-01 UTC and were retained in the analysis dataset."
            ),
            (
                f"Annual salary is disclosed for "
                f"{job_postings_analysis['salary_year_avg'].notna().sum():,} postings "
                f"({job_postings_analysis['salary_year_avg'].notna().mean() * 100:.2f}%)."
            ),
            (
                f"Hourly salary is disclosed for "
                f"{job_postings_analysis['salary_hour_avg'].notna().sum():,} postings "
                f"({job_postings_analysis['salary_hour_avg'].notna().mean() * 100:.2f}%)."
            ),
            (
                f"job_location is missing for "
                f"{job_postings_analysis['job_location'].isna().sum():,} postings "
                f"({job_postings_analysis['job_location'].isna().mean() * 100:.2f}%)."
            ),
            (
                f"job_location contains "
                f"{job_postings_analysis['job_location'].nunique(dropna=True):,} unique non-missing "
                "raw text values and requires parsing before city-level mapping."
            ),
            (
                f"company_id has "
                f"{job_postings_analysis['company_id'].isna().sum():,} missing values."
            ),
            (
                "The work-from-home flag and 'Anywhere' location do not guarantee "
                "that a candidate living in Poland is eligible for legal employment."
            ),
            (
                "The dataset represents observed job postings from its included sources; "
                "results should not be interpreted as a complete census of the global labour market."
            ),
        ],
        "analysis_implication": [
            (
                "Retain the full source dataset and document the UTC date boundary condition "
                "when describing the temporal scope."
            ),
            (
                "Interpret annual-salary statistics only for postings with disclosed annual salary; "
                "do not generalize directly to all postings."
            ),
            (
                "Do not use hourly salary as the primary pay metric because of very limited coverage."
            ),
            (
                "Report map coverage and exclude missing locations from city-level geocoding."
            ),
            (
                "Audit formats, parse locations carefully, geocode only validated unique locations, "
                "and report the unmatched share."
            ),
            (
                "Company-level analyses can use the fact table's company_id without missing-value filtering, "
                "but company metadata still requires separate completeness checks."
            ),
            (
                "Describe remote roles as potentially accessible rather than guaranteed cross-border opportunities."
            ),
            (
                "Frame results as patterns in the observed dataset, not as definitive facts about every job posting globally."
            ),
        ],
    }
)

display(data_limitations)

,area,observation,analysis_implication
0,Temporal scope,90 postings are dated before 2024-01-01 UTC an...,Retain the full source dataset and document th...
1,Salary coverage,"Annual salary is disclosed for 20,335 postings...",Interpret annual-salary statistics only for po...
2,Hourly salary coverage,"Hourly salary is disclosed for 10,076 postings...",Do not use hourly salary as the primary pay me...
3,Location completeness,"job_location is missing for 1,373 postings (0....",Report map coverage and exclude missing locati...
4,Location standardization,"job_location contains 16,360 unique non-missin...","Audit formats, parse locations carefully, geoc..."
5,Company mapping,company_id has 0 missing values.,Company-level analyses can use the fact table'...
6,Remote-work interpretation,The work-from-home flag and 'Anywhere' locatio...,Describe remote roles as potentially accessibl...
7,Dataset representation,The dataset represents observed job postings f...,Frame results as patterns in the observed data...
